In [18]:
!pip install -q --upgrade qiskit qiskit-aer qiskit-ibm-runtime

In [19]:
# Block 1 — Imports
import numpy as np
import random
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Measure
from qiskit_aer import AerSimulator
from qiskit_aer.noise import (
    NoiseModel,
    depolarizing_error,
    thermal_relaxation_error,
    ReadoutError,
)
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from kaggle_secrets import UserSecretsClient
import warnings
warnings.filterwarnings("ignore", module="samplomatic")

In [20]:
# Block 2 — IBM Backend connection
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("IBM_QUANTUM_TOKEN")

service = QiskitRuntimeService(token=token)

backend = service.least_busy(simulator=False, operational=True, min_num_qubits=3)
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
    translation_method="translator"
)

print("backend :", backend.name)
print("qubits  :", backend.num_qubits)
print("instance:", service.active_instance())

qiskit_runtime_service._discover_account:WARNING:2026-05-12 13:25:52,355: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-05-12 13:25:55,009: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-05-12 13:25:55,584: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-05-12 13:25:56,964: Using instance: open-instance, plan: open


backend : ibm_kingston
qubits  : 156
instance: crn:v1:bluemix:public:quantum-computing:us-east:a/bce7e35332094c958667013546978c93:b9d57103-aa40-4115-9506-8bbfed29d286::


In [21]:
# Block 3 — Grover circuits (2 qubits, oracle marks |11>)
# Correct : standard Grover — CZ oracle + one diffusion step
# Buggy   : same + one random X (bit-flip) injected after diffusion

N_QUBITS = 2
SHOTS_GR = 8192

def build_grover_2q(inject_x_on=None):
    """Build 2-qubit Grover circuit; optionally inject a bit-flip bug."""
    qc = QuantumCircuit(N_QUBITS, N_QUBITS)
    qc.h([0, 1])           # uniform superposition
    qc.cz(0, 1)            # oracle: phase-flip |11>
    qc.h([0, 1])           # diffusion start
    qc.x([0, 1])
    qc.h(1); qc.cx(0, 1); qc.h(1)
    qc.x([0, 1])
    qc.h([0, 1])           # diffusion end
    if inject_x_on is not None:
        qc.x(inject_x_on)  # BUG: bit-flip error
    qc.measure([0, 1], [0, 1])
    return qc

bug_qubit      = random.choice([0, 1])
grover_correct = build_grover_2q()
grover_buggy   = build_grover_2q(inject_x_on=bug_qubit)

print("Correct Grover circuit:")
print(grover_correct.draw(output="text", fold=-1))
print(f"\nBuggy Grover circuit (X injected on q{bug_qubit}):")
print(grover_buggy.draw(output="text", fold=-1))

Correct Grover circuit:
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤┌───┐└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 

Buggy Grover circuit (X injected on q0):
     ┌───┐   ┌───┐┌───┐          ┌───┐┌───┐┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ X ├───────■──┤ X ├┤ H ├┤ X ├┤M├───
     ├───┤ │ ├───┤├───┤┌───┐┌─┴─┐├───┤├───┤├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├─╫─┤M├
     └───┘   └───┘└───┘└───┘└───┘└───┘└───┘└───┘ ║ └╥┘
c: 2/════════════════════════════════════════════╩══╩═
                                                 0  1 


In [22]:
# Block 4 — Helper utilities

def counts_to_probs(counts, nqubits):
    """Convert Qiskit counts dict → probability vector of length 2^nqubits."""
    total = sum(counts.values())
    probs = np.zeros(2 ** nqubits, dtype=float)
    for bitstring, count in counts.items():
        probs[int(bitstring, 2)] = count / total
    return probs

def sampler_result_to_counts(result_entry):
    """Extract bitstring counts from a SamplerV2 pub result entry."""
    try:
        return result_entry.data.c.get_counts()
    except AttributeError:
        pass
    try:
        reg_name = list(result_entry.data.keys())[0]
        return result_entry.data[reg_name].get_counts()
    except Exception:
        pass
    raise RuntimeError("Cannot extract counts from SamplerV2 result entry.")

def get_usage_seconds(service):
    """Return (used_seconds, remaining_seconds) for this service instance."""
    usage     = service.usage()
    used      = float(usage.get("usage_consumed_seconds", 0.0))
    remaining = usage.get("usage_remaining_seconds")
    if remaining is None:
        limit     = usage.get("usage_limit_seconds") or usage.get("usage_allocation_seconds")
        remaining = float(limit - used) if limit is not None else None
    return used, remaining

def count_1q_2q_gates(qc):
    """Count single- and two-qubit gates, ignoring measurements."""
    n_1q = n_2q = 0
    for ci in qc.data:
        if isinstance(ci.operation, Measure):
            continue
        if   len(ci.qubits) == 1: n_1q += 1
        elif len(ci.qubits) == 2: n_2q += 1
    return n_1q, n_2q

In [23]:
# Block 5 — Ideal (noiseless) reference distribution

simulator_ideal = AerSimulator()

t_ideal         = transpile(grover_correct, simulator_ideal)
counts_ideal_gr = simulator_ideal.run(t_ideal, shots=SHOTS_GR).result().get_counts()
probs_ideal_gr  = counts_to_probs(counts_ideal_gr, N_QUBITS)

print("Ideal distribution:")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_ideal_gr[idx] > 1e-4:
        print(f"  |{b}>  {probs_ideal_gr[idx]:.4f}")

Ideal distribution:
  |11>  1.0000


In [34]:
# Block 6 — Hardware-accurate noise estimator with per-gate effective errors
#
# This version computes, for each physical qubit q actually used:
#   e_1q_eff(q) = combined error from all 1Q gates applied to q
#   e_2q_eff(q) = combined error from all 2Q gates touching q
#   e_ro(q)     = readout error on q
#   e_total(q)  = 1 - (1 - e_1q_eff(q)) (1 - e_2q_eff(q)) (1 - e_ro(q))
#
# It also builds a noise model using backend calibration data.

from collections import defaultdict

def get_readout_error_details(props, q):
    try:
        ro = float(props.readout_error(q))
        p1_g0 = 0.8 * ro
        p0_g1 = ro
        return ro, p0_g1, p1_g0
    except Exception:
        pass

    try:
        qprops = props.qubits[q]
        p0_g1 = next((float(p.value) for p in qprops if p.name == "prob_meas0_prep1"), None)
        p1_g0 = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), None)
        if p0_g1 is not None and p1_g0 is not None:
            ro = 0.5 * (p0_g1 + p1_g0)
            return ro, p0_g1, p1_g0
    except Exception:
        pass

    return 0.01, 0.01, 0.01


def analyze_effective_errors_from_transpiled_circuit(tqc, backend):
    props = backend.properties()

    oneq_survival = defaultdict(lambda: 1.0)
    twoq_survival = defaultdict(lambda: 1.0)
    gate_time_1q_sum = defaultdict(float)
    gate_time_2q_sum = defaultdict(float)

    gate_log_1q = defaultdict(list)
    gate_log_2q = defaultdict(list)

    used_qubits = set()

    for ci in tqc.data:
        inst = ci.operation
        qargs = ci.qubits
        name = inst.name

        if name in ["measure", "barrier"]:
            continue

        phys_qubits = [tqc.find_bit(q).index for q in qargs]

        if len(phys_qubits) == 1:
            q = phys_qubits[0]
            used_qubits.add(q)

            try:
                e = float(props.gate_error(name, [q]))
            except Exception:
                try:
                    e = float(props.gate_error("sx", [q]))
                except Exception:
                    e = 0.0

            try:
                gt = float(props.gate_length(name, [q]))
            except Exception:
                try:
                    gt = float(props.gate_length("sx", [q]))
                except Exception:
                    gt = 35e-9

            oneq_survival[q] *= (1.0 - e)
            gate_time_1q_sum[q] += gt
            gate_log_1q[q].append((name, e, gt))

        elif len(phys_qubits) == 2:
            q0, q1 = phys_qubits
            used_qubits.update([q0, q1])

            try:
                e2 = float(props.gate_error(name, [q0, q1]))
            except Exception:
                try:
                    e2 = float(props.gate_error(name, [q1, q0]))
                except Exception:
                    try:
                        e2 = float(props.gate_error("cx", [q0, q1]))
                    except Exception:
                        e2 = 0.0

            try:
                gt2 = float(props.gate_length(name, [q0, q1]))
            except Exception:
                try:
                    gt2 = float(props.gate_length(name, [q1, q0]))
                except Exception:
                    try:
                        gt2 = float(props.gate_length("cx", [q0, q1]))
                    except Exception:
                        gt2 = 300e-9

            twoq_survival[q0] *= (1.0 - e2)
            twoq_survival[q1] *= (1.0 - e2)

            gate_time_2q_sum[q0] += gt2
            gate_time_2q_sum[q1] += gt2

            gate_log_2q[q0].append((f"{name}{(q0, q1)}", e2, gt2))
            gate_log_2q[q1].append((f"{name}{(q0, q1)}", e2, gt2))

    analysis = {}
    for q in sorted(used_qubits):
        try:
            t1 = float(props.t1(q))
        except Exception:
            t1 = 100e-6

        try:
            t2 = float(props.t2(q))
        except Exception:
            t2 = 80e-6

        t2 = min(t2, 2 * t1)

        e_1q_eff = 1.0 - oneq_survival[q]
        e_2q_eff = 1.0 - twoq_survival[q]

        e_ro, p0_g1, p1_g0 = get_readout_error_details(props, q)

        e_total = 1.0 - (1.0 - e_1q_eff) * (1.0 - e_2q_eff) * (1.0 - e_ro)

        analysis[q] = {
            "T1": t1,
            "T2": t2,
            "gate_time_1q": gate_time_1q_sum[q],
            "gate_time_2q": gate_time_2q_sum[q],
            "e_1q_eff": e_1q_eff,
            "e_2q_eff": e_2q_eff,
            "e_ro": e_ro,
            "e_total": e_total,
            "P(0|1)": p0_g1,
            "P(1|0)": p1_g0,
            "gates_1q": gate_log_1q[q],
            "gates_2q": gate_log_2q[q],
        }

    return analysis


def build_hardware_noise_model_from_analysis(backend, analysis):
    noise_model = NoiseModel()

    for q, d in analysis.items():
        t1 = d["T1"]
        t2 = d["T2"]

        gate_time_1q = d["gate_time_1q"] if d["gate_time_1q"] > 0 else 35e-9
        gate_time_2q = d["gate_time_2q"] if d["gate_time_2q"] > 0 else 300e-9

        e_1q_eff = d["e_1q_eff"]
        e_ro = d["e_ro"]
        p0_g1 = d["P(0|1)"]
        p1_g0 = d["P(1|0)"]

        therm_1q = thermal_relaxation_error(t1=t1, t2=t2, time=gate_time_1q)
        depo_1q = depolarizing_error(min(max(e_1q_eff, 0.0), 0.999999), 1)
        combined_1q = therm_1q.compose(depo_1q)

        for gate in ["sx", "x", "id", "rz", "h"]:
            try:
                noise_model.add_quantum_error(combined_1q, gate, [q])
            except Exception:
                pass

        ro_matrix = [
            [1 - p1_g0, p1_g0],
            [p0_g1, 1 - p0_g1],
        ]
        try:
            noise_model.add_readout_error(ReadoutError(ro_matrix), [q])
        except Exception:
            pass

    qubits_used = sorted(analysis.keys())
    for i in range(len(qubits_used)):
        for j in range(i + 1, len(qubits_used)):
            q0, q1 = qubits_used[i], qubits_used[j]

            e2_q0 = analysis[q0]["e_2q_eff"]
            e2_q1 = analysis[q1]["e_2q_eff"]
            e2_pair = max(e2_q0, e2_q1)

            gate_time_2q = max(
                analysis[q0]["gate_time_2q"] if analysis[q0]["gate_time_2q"] > 0 else 300e-9,
                analysis[q1]["gate_time_2q"] if analysis[q1]["gate_time_2q"] > 0 else 300e-9,
            )

            therm_q0 = thermal_relaxation_error(
                t1=analysis[q0]["T1"], t2=analysis[q0]["T2"], time=gate_time_2q
            )
            therm_q1 = thermal_relaxation_error(
                t1=analysis[q1]["T1"], t2=analysis[q1]["T2"], time=gate_time_2q
            )
            therm_2q = therm_q0.expand(therm_q1)
            depo_2q = depolarizing_error(min(max(e2_pair, 0.0), 0.999999), 2)
            combined_2q = therm_2q.compose(depo_2q)

            for gate in ["cx", "cz", "ecr"]:
                for pair in ([q0, q1], [q1, q0]):
                    try:
                        noise_model.add_quantum_error(combined_2q, gate, pair)
                    except Exception:
                        pass

    return noise_model


def print_calibration_table(calib_table, backend_name=""):
    if backend_name:
        print(f"\nCalibration Data ({backend_name})")
    print(f" {'Qubit':>5} {'T1 (µs)':>10} {'T2 (µs)':>10} {'1q_t(ns)':>10} {'2q_t(ns)':>10} "
          f"{'e_1q_eff':>12} {'e_2q_eff':>12} {'e_ro':>10} {'e_total':>10}")
    print(" " + "-" * 103)

    for q, d in sorted(calib_table.items()):
        print(
            f" q{q:<4} "
            f"{d['T1']*1e6:>10.2f} "
            f"{d['T2']*1e6:>10.2f} "
            f"{d['gate_time_1q']*1e9:>10.2f} "
            f"{d['gate_time_2q']*1e9:>10.2f} "
            f"{d['e_1q_eff']:>12.6e} "
            f"{d['e_2q_eff']:>12.6e} "
            f"{d['e_ro']:>10.6f} "
            f"{d['e_total']:>10.6f}"
        )
    print()

In [35]:
# Block 7 — Identify physical qubits after transpilation
# + compute per-gate effective errors
# + build hardware-accurate noise model from those effective errors

tqc_hw = pm.run(grover_correct)

try:
    layout_gr = tqc_hw.layout.final_layout
    used_physical_qubits = sorted({
        layout_gr[grover_correct.qubits[q]] for q in range(N_QUBITS)
    })
except Exception:
    used_physical_qubits = list(range(N_QUBITS))

print(f"Physical qubits used on {backend.name}: {used_physical_qubits}")

print("\nExtracting calibration data from IBM backend ...")

# Analyze the ACTUAL transpiled circuit gate-by-gate
calib_table = analyze_effective_errors_from_transpiled_circuit(tqc_hw, backend)

# Keep only the qubits actually used
calib_table = {q: calib_table[q] for q in used_physical_qubits if q in calib_table}

print_calibration_table(calib_table, backend_name=backend.name)

print("Combination equation used:")
print("  e_total(q) = 1 - (1 - e_1q_eff(q)) (1 - e_2q_eff(q)) (1 - e_ro(q))")

noise_model_hw = build_hardware_noise_model_from_analysis(backend, calib_table)
simulator_est = AerSimulator(noise_model=noise_model_hw)

Physical qubits used on ibm_kingston: [0, 1]

Extracting calibration data from IBM backend ...

Calibration Data (ibm_kingston)
 Qubit    T1 (µs)    T2 (µs)   1q_t(ns)   2q_t(ns)     e_1q_eff     e_2q_eff       e_ro    e_total
 -------------------------------------------------------------------------------------------------------
 q0        329.51     328.27      96.00     136.00 6.607276e-04 3.362714e-03   0.011597   0.015571
 q1        315.95     631.91      96.00     136.00 7.739142e-04 3.362714e-03   0.106689   0.110382

Combination equation used:
  e_total(q) = 1 - (1 - e_1q_eff(q)) (1 - e_2q_eff(q)) (1 - e_ro(q))


In [26]:
# Block 8 — Run Grover circuit under the hardware noise estimator

tqc_est        = transpile(grover_correct, simulator_est)
counts_est_cor = simulator_est.run(tqc_est, shots=SHOTS_GR).result().get_counts()
probs_est_cor  = counts_to_probs(counts_est_cor, N_QUBITS)

tqc_est_bug    = transpile(grover_buggy, simulator_est)
counts_est_bug = simulator_est.run(tqc_est_bug, shots=SHOTS_GR).result().get_counts()
probs_est_bug  = counts_to_probs(counts_est_bug, N_QUBITS)

print("Noise-Estimator distribution (correct circuit):")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_est_cor[idx] > 1e-4:
        print(f"  |{b}>  {probs_est_cor[idx]:.4f}")

Noise-Estimator distribution (correct circuit):
  |00>  0.0040
  |01>  0.1089
  |10>  0.0129
  |11>  0.8741


In [27]:
# Block 9 — Run Grover circuit on real IBM hardware
# Also print:
#   - usage before the job
#   - usage after the job
#   - seconds consumed by this circuit job
#   - seconds remaining

print(f"Running on real hardware: {backend.name} ...")

# ----- usage before -----
usage_before = service.usage()
used_before = float(usage_before.get("usage_consumed_seconds", 0.0))
remaining_before = usage_before.get("usage_remaining_seconds")

print(f"Before job  -> used: {used_before:.1f}s", end="")
if remaining_before is not None:
    print(f" | remaining: {remaining_before:.1f}s")
else:
    print()

# ----- submit job -----
tqc_hw_cor = pm.run(grover_correct)
tqc_hw_bug = pm.run(grover_buggy)

sampler = Sampler(mode=backend)
job = sampler.run([tqc_hw_cor, tqc_hw_bug], shots=SHOTS_GR)
result = job.result()

# ----- extract counts -----
counts_hw_cor = sampler_result_to_counts(result[0])
counts_hw_bug = sampler_result_to_counts(result[1])
probs_hw_cor  = counts_to_probs(counts_hw_cor, N_QUBITS)
probs_hw_bug  = counts_to_probs(counts_hw_bug, N_QUBITS)

# ----- usage after -----
usage_after = service.usage()
used_after = float(usage_after.get("usage_consumed_seconds", 0.0))
remaining_after = usage_after.get("usage_remaining_seconds")

job_secs = used_after - used_before

print(f"After job   -> used: {used_after:.1f}s", end="")
if remaining_after is not None:
    print(f" | remaining: {remaining_after:.1f}s")
else:
    print()

print(f"Job usage   -> this execution consumed: {job_secs:.1f}s")

print("\nReal Hardware distribution (correct circuit):")
for idx in range(2 ** N_QUBITS):
    b = format(idx, f"0{N_QUBITS}b")
    if probs_hw_cor[idx] > 1e-4:
        print(f"  |{b}>  {probs_hw_cor[idx]:.4f}")

Running on real hardware: ibm_kingston ...
Before job  -> used: 101.0s | remaining: 499.0s
After job   -> used: 107.0s | remaining: 493.0s
Job usage   -> this execution consumed: 6.0s

Real Hardware distribution (correct circuit):
  |00>  0.0016
  |01>  0.0170
  |10>  0.0128
  |11>  0.9686


In [28]:
# Block 10 — Per-qubit bit-flip error rate
#
# For each qubit i (marginalised over all other qubits):
#   d_i = 0 if P_ideal(qubit_i = 1) < 0.5 else 1    ← ideal dominant bit
#   flip_rate_i = P_measured(qubit_i ≠ d_i)

def marginal_p1(probs, nqubits, q):
    """Return P(qubit q = 1) marginalised over all other qubits."""
    p1 = 0.0
    for idx in range(2 ** nqubits):
        if (idx >> q) & 1:
            p1 += probs[idx]
    return p1

def per_qubit_bitflip_rate(probs_measured, probs_ideal, nqubits):
    flip_rates = {}
    for q in range(nqubits):
        p1_ideal = marginal_p1(probs_ideal,    nqubits, q)
        p1_meas  = marginal_p1(probs_measured, nqubits, q)
        d = 1 if p1_ideal >= 0.5 else 0       # ideal dominant bit
        if d == 0:
            flip_rates[q] = float(p1_meas)         # excess 1s = flips
        else:
            flip_rates[q] = float(1.0 - p1_meas)   # excess 0s = flips
    return flip_rates

In [29]:
# Block 11 — Compare per-qubit flip-rate ratios: estimator vs hardware
#
# ratio_q = est_flip_rate_q / hw_flip_rate_q
# |ratio_q - 1| <= 0.30  →  ✓ MATCH
# otherwise              →  ✗ MISMATCH (estimator needs refinement)

def compare_flip_rate_ratios(est_rates, hw_rates, nqubits, tolerance=0.30):
    print(f"\n  {'Qubit':>5}  {'Est. Flip':>12}  {'HW Flip':>12}  "
          f"{'Ratio (E/H)':>13}  {'|Ratio-1|':>11}  {'Status':>12}")
    print("  " + "-" * 72)
    all_match = True
    for q in range(nqubits):
        er = est_rates.get(q, 0.0)
        hr = hw_rates.get(q, 0.0)
        if hr > 1e-8:
            ratio = er / hr
        elif er > 1e-8:
            ratio = np.inf
        else:
            ratio = 1.0                    # both ~0 → perfect match
        deviation = abs(ratio - 1.0)
        match     = deviation <= tolerance and not np.isinf(ratio)
        if not match:
            all_match = False
        flag = "✓ MATCH" if match else "✗ MISMATCH"
        print(f"  q{q:<4}  {er:>12.6f}  {hr:>12.6f}  {ratio:>13.4f}  "
              f"{deviation:>11.4f}  {flag:>12}")
    print("  " + "-" * 72)
    if all_match:
        print(f"  OVERALL VERDICT : Noise estimator is ACCURATE")
        print(f"                    All per-qubit ratios within ±{tolerance*100:.0f}% of hardware")
    else:
        print(f"  OVERALL VERDICT : Noise estimator NEEDS REFINEMENT")
        print(f"                    Some per-qubit ratios exceed ±{tolerance*100:.0f}% deviation")
    print()
    return all_match

In [30]:
# Block 12 — Compute per-qubit bit-flip error rates
#             noise estimator vs real hardware vs ideal

flip_est = per_qubit_bitflip_rate(probs_est_cor, probs_ideal_gr, N_QUBITS)
flip_hw  = per_qubit_bitflip_rate(probs_hw_cor,  probs_ideal_gr, N_QUBITS)

print("Per-Qubit Bit-Flip Error Rates")
print("=" * 50)
print(f"  {'Qubit':>5}  {'Noise Estimator':>16}  {'Real Hardware':>14}")
print("  " + "-" * 42)
for q in range(N_QUBITS):
    print(f"  q{q:<4}  {flip_est[q]:>16.6f}  {flip_hw[q]:>14.6f}")
print()

Per-Qubit Bit-Flip Error Rates
  Qubit   Noise Estimator   Real Hardware
  ------------------------------------------
  q0             0.016968        0.014404
  q1             0.112915        0.018555



In [31]:
# Block 13 — Noise estimator accuracy assessment

print("=" * 74)
print("  NOISE ESTIMATOR ACCURACY ASSESSMENT  (tolerance = ±30%)")
print("=" * 74)

estimator_accurate = compare_flip_rate_ratios(
    flip_est, flip_hw, nqubits=N_QUBITS, tolerance=0.30
)

  NOISE ESTIMATOR ACCURACY ASSESSMENT  (tolerance = ±30%)

  Qubit     Est. Flip       HW Flip    Ratio (E/H)    |Ratio-1|        Status
  ------------------------------------------------------------------------
  q0         0.016968      0.014404         1.1780       0.1780       ✓ MATCH
  q1         0.112915      0.018555         6.0855       5.0855    ✗ MISMATCH
  ------------------------------------------------------------------------
  OVERALL VERDICT : Noise estimator NEEDS REFINEMENT
                    Some per-qubit ratios exceed ±30% deviation



In [32]:
# Block 14 — Per-qubit effective error using actual transpiled gates
# Includes:
#   e_1q_eff(q)  from all 1Q gates applied to qubit q in the transpiled circuit
#   e_2q_eff(q)  from all 2Q gates touching qubit q in the transpiled circuit
#   e_ro(q)      from backend readout calibration
#
# Combination equation:
#   e_total(q) = 1 - (1 - e_1q_eff(q)) (1 - e_2q_eff(q)) (1 - e_ro(q))

from collections import defaultdict

def get_readout_error(props, q):
    try:
        return float(props.readout_error(q))
    except Exception:
        pass
    try:
        qprops = props.qubits[q]
        p01 = next((float(p.value) for p in qprops if p.name == "prob_meas0_prep1"), None)
        p10 = next((float(p.value) for p in qprops if p.name == "prob_meas1_prep0"), None)
        if p01 is not None and p10 is not None:
            return 0.5 * (p01 + p10)
    except Exception:
        pass
    return 0.0

def per_qubit_effective_errors_from_transpiled_circuit(tqc, backend):
    props = backend.properties()

    oneq_survival = defaultdict(lambda: 1.0)
    twoq_survival = defaultdict(lambda: 1.0)
    gate_log_1q   = defaultdict(list)
    gate_log_2q   = defaultdict(list)

    for ci in tqc.data:
        inst = ci.operation
        qargs = ci.qubits
        name = inst.name

        if name in ["measure", "barrier"]:
            continue

        phys_qubits = [tqc.find_bit(q).index for q in qargs]

        # 1-qubit gate
        if len(phys_qubits) == 1:
            q = phys_qubits[0]
            try:
                e = float(props.gate_error(name, [q]))
            except Exception:
                e = 0.0
            oneq_survival[q] *= (1.0 - e)
            gate_log_1q[q].append((name, e))

        # 2-qubit gate
        elif len(phys_qubits) == 2:
            q0, q1 = phys_qubits
            try:
                e2 = float(props.gate_error(name, [q0, q1]))
            except Exception:
                try:
                    e2 = float(props.gate_error(name, [q1, q0]))
                except Exception:
                    e2 = 0.0

            twoq_survival[q0] *= (1.0 - e2)
            twoq_survival[q1] *= (1.0 - e2)

            gate_log_2q[q0].append((f"{name}{(q0, q1)}", e2))
            gate_log_2q[q1].append((f"{name}{(q0, q1)}", e2))

    used_qubits = sorted(set(list(oneq_survival.keys()) + list(twoq_survival.keys())))

    perq = {}
    for q in used_qubits:
        e_1q_eff = 1.0 - oneq_survival[q]
        e_2q_eff = 1.0 - twoq_survival[q]
        e_ro     = get_readout_error(props, q)

        e_total = 1.0 - (1.0 - e_1q_eff) * (1.0 - e_2q_eff) * (1.0 - e_ro)

        perq[q] = {
            "e_1q_eff": e_1q_eff,
            "e_2q_eff": e_2q_eff,
            "e_ro": e_ro,
            "e_total": e_total,
            "gates_1q": gate_log_1q[q],
            "gates_2q": gate_log_2q[q],
        }

    return perq

# Use the transpiled correct circuit
tqc_hw_cor = pm.run(grover_correct)
perq_eff   = per_qubit_effective_errors_from_transpiled_circuit(tqc_hw_cor, backend)

virt_to_phys = {v: p for v, p in enumerate(sorted(used_physical_qubits))}

print(f"Per-Qubit Effective Error Summary  (bug injected on q{bug_qubit})")
print("=" * 90)
print(f"  {'VQ':>2}  {'PQ':>3}  {'e_1q_eff':>12}  {'e_2q_eff':>12}  {'e_ro':>10}  {'e_total':>12}  {'Obs flip':>10}")
print("-" * 90)

for vq in range(N_QUBITS):
    pq  = virt_to_phys[vq]
    d   = perq_eff[pq]
    obs = flip_hw[vq]
    tag = "  <- BUG QUBIT" if vq == bug_qubit else ""
    print(f"  q{vq}  q{pq:>2}  {d['e_1q_eff']:>12.6f}  {d['e_2q_eff']:>12.6f}  "
          f"{d['e_ro']:>10.6f}  {d['e_total']:>12.6f}  {obs:>10.6f}{tag}")

print("=" * 90)

safest   = min(range(N_QUBITS), key=lambda q: perq_eff[virt_to_phys[q]]["e_total"])
riskiest = max(range(N_QUBITS), key=lambda q: perq_eff[virt_to_phys[q]]["e_total"])

print(f"\n  Safest   qubit → q{safest}  "
      f"(e_total = {perq_eff[virt_to_phys[safest]]['e_total']:.6f}, "
      f"observed flip = {flip_hw[safest]:.6f})")
print(f"  Riskiest qubit → q{riskiest}  "
      f"(e_total = {perq_eff[virt_to_phys[riskiest]]['e_total']:.6f}, "
      f"observed flip = {flip_hw[riskiest]:.6f})")

print("\nCombination equation used:")
print("  e_total(q) = 1 - (1 - e_1q_eff(q)) (1 - e_2q_eff(q)) (1 - e_ro(q))")

Per-Qubit Effective Error Summary  (bug injected on q0)
  VQ   PQ      e_1q_eff      e_2q_eff        e_ro       e_total    Obs flip
------------------------------------------------------------------------------------------
  q0  q 0      0.000661      0.003363    0.011597      0.015571    0.014404  <- BUG QUBIT
  q1  q 1      0.000774      0.003363    0.106689      0.110382    0.018555

  Safest   qubit → q0  (e_total = 0.015571, observed flip = 0.014404)
  Riskiest qubit → q1  (e_total = 0.110382, observed flip = 0.018555)

Combination equation used:
  e_total(q) = 1 - (1 - e_1q_eff(q)) (1 - e_2q_eff(q)) (1 - e_ro(q))


In [33]:
# Block 15 — Full probability distribution table
#             Ideal | Noise Estimator | HW correct | HW buggy

def print_dist_table(probs_ideal, probs_est, probs_hw_cor, probs_hw_bug, nqubits):
    print("\nGrover Circuit — Full Probability Distribution")
    print("=" * 74)
    print(f"  {'State':>6}  {'Ideal':>10}  {'Noise Est.':>12}  "
          f"{'HW correct':>12}  {'HW buggy':>10}  {'Est vs HW':>10}")
    print("  " + "-" * 68)
    for idx in range(2 ** nqubits):
        b     = format(idx, f"0{nqubits}b")
        delta = abs(probs_est[idx] - probs_hw_cor[idx])
        flag  = "  ←" if delta > 0.05 else ""
        print(f"  {b:>6}  {probs_ideal[idx]:>10.4f}  {probs_est[idx]:>12.4f}  "
              f"{probs_hw_cor[idx]:>12.4f}  {probs_hw_bug[idx]:>10.4f}  "
              f"{delta:>10.4f}{flag}")
    print("  " + "-" * 68)
    mad = np.mean([abs(probs_est[i] - probs_hw_cor[i]) for i in range(2**nqubits)])
    print(f"  Mean Absolute Deviation (Est vs HW correct): {mad:.4f}")
    if   mad <= 0.05: label = "GOOD           (MAD ≤ 0.05)"
    elif mad <= 0.10: label = "ACCEPTABLE     (MAD ≤ 0.10)"
    else:             label = "POOR           (MAD > 0.10 — refine the estimator)"
    print(f"  Distribution accuracy  : {label}")
    print()

print_dist_table(probs_ideal_gr, probs_est_cor, probs_hw_cor, probs_hw_bug, N_QUBITS)


Grover Circuit — Full Probability Distribution
   State       Ideal    Noise Est.    HW correct    HW buggy   Est vs HW
  --------------------------------------------------------------------
      00      0.0000        0.0040        0.0016      0.0190      0.0024
      01      0.0000        0.1089        0.0170      0.0042      0.0919  ←
      10      0.0000        0.0129        0.0128      0.9634      0.0001
      11      1.0000        0.8741        0.9686      0.0134      0.0945  ←
  --------------------------------------------------------------------
  Mean Absolute Deviation (Est vs HW correct): 0.0472
  Distribution accuracy  : GOOD           (MAD ≤ 0.05)

